# 2. Extração de Features Manuais
Este notebook aplica o pipeline de extração de características geométricas, inerciais, de cor e textura (GLCM) sobre as imagens segmentadas e exporta a matriz X e o vetor y em formato CSV.

In [14]:
import cv2
import numpy as np
import pandas as pd
import os
import glob
from skimage.feature import graycomatrix, graycoprops

def load_image(filepath):
    img = cv2.imread(filepath)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def segment_otsu(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def extract_features(img_rgb, mask):
    features = {}
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
        
    c = max(contours, key=cv2.contourArea)
    
    area = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)
    features['area'] = area
    features['perimeter'] = perimeter
    features['circularity'] = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

    moments = cv2.moments(c)
    hu_moments = cv2.HuMoments(moments).flatten()
    for i, hu in enumerate(hu_moments):
        features[f'hu_moment_{i}'] = -np.sign(hu) * np.log10(np.abs(hu)) if hu != 0 else 0

    mean_val = cv2.mean(img_rgb, mask=mask)
    features['mean_R'] = mean_val[0]
    features['mean_G'] = mean_val[1]
    features['mean_B'] = mean_val[2]

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray_masked = cv2.bitwise_and(gray, gray, mask=mask)
    glcm = graycomatrix(gray_masked, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    
    features['glcm_contrast'] = graycoprops(glcm, 'contrast')[0, 0]
    features['glcm_correlation'] = graycoprops(glcm, 'correlation')[0, 0]
    features['glcm_energy'] = graycoprops(glcm, 'energy')[0, 0]
    features['glcm_homogeneity'] = graycoprops(glcm, 'homogeneity')[0, 0]

    return features

## Processamento em Lote
Iteração nas pastas do dataset para processar as imagens (limitado a 200 por classe para balanceamento) e exportação dos dados.

In [15]:
diretorio_base = "/home/akdag/Desktop/vis_computacional/dataset"
limite_por_classe = 200
dados_tabela = []

pastas_alvo = [
    ("good_quality_fruits/Apple_Good", "fresh"),
    ("good_quality_fruits/Banana_Good", "fresh"),
    ("good_quality_fruits/Orange_Good", "fresh"),
    ("bad_quality_fruits/Apple_Bad", "rotten"),
    ("bad_quality_fruits/Banana_Bad", "rotten"),
    ("bad_quality_fruits/Orange_Bad", "rotten")
]

for pasta_relativa, label_classe in pastas_alvo:
    caminho_completo = os.path.join(diretorio_base, pasta_relativa)
    arquivos = glob.glob(os.path.join(caminho_completo, '*.[jJ][pP]*'))
    contador = 0
    
    for arquivo in arquivos:
        if contador >= limite_por_classe:
            break
            
        img_original = load_image(arquivo)
        if img_original is None:
            continue
        
        m_otsu = segment_otsu(img_original)
        features = extract_features(img_original, m_otsu)
        
        if features is not None:
            features['label'] = label_classe
            dados_tabela.append(features)
            contador += 1

df = pd.DataFrame(dados_tabela)
X = df.drop(columns=['label'])
y = df['label']

os.makedirs('outputs', exist_ok=True)
X.to_csv('outputs/X.csv', index=False)
y.to_csv('outputs/y.csv', index=False)